In [39]:
from openai import OpenAI
import gradio as gr
import glob
from pathlib import Path
from dotenv import load_dotenv
import os

In [41]:
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

GOOGLE_API_KEY is not None or GOOGLE_API_KEY != ''

True

In [69]:
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=GOOGLE_API_KEY)

In [61]:
# Use file handling to read the knowledge from folders to a python dict

knowledge = {}

knowledge_base_path = "knowledge-base/**/*.md"

files = glob.glob(
    knowledge_base_path,
    recursive=True
)

print(f"Found {len(files)} files in knowledge base")

for file in files:
    file = Path(file)
    filename = file.stem
    with open(file, 'r', encoding='utf-8') as f:
        knowledge[filename] = f.read()
    for word in filename.split():
        with open(file, 'r', encoding='utf-8') as f:
                knowledge[word] = f.read()
    
print("Converted knowledge base to python dict")


Found 76 files in knowledge base
Converted knowledge base to python dict


In [68]:
# Define the first part of the system prompt

SYSTEM_PROMPT_PREFIX = """
You are a knowledge assistant for an insurance company named insurellm who responds to user queries. 
You might be given additional context regarding the user query. However, it might be possible that you don't receive a context at times.
Your response should be crisp, clear, short and direct.
Do not respond to questions with hallucinations if you do not have the context. Gracefully say so. 
If you have the context, respond.

The context is as so - \n\n
"""

In [72]:
def get_relevant_context(message):
    message = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    return "".join([knowledge[name] for name in message.split() if name in knowledge])

In [75]:
def chat(message, history):
    system_prompt = SYSTEM_PROMPT_PREFIX + get_relevant_context(message)
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content":message}]
    response = gemini.chat.completions.create(
        model="gemini-3.5-flash-lite",
        messages=messages,
    )

    return response.choices[0].message.content

In [76]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
